In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer — Metadata-Driven Cleansing & Upsert
# MAGIC Reads a Bronze table, applies standard cleansing, dedupes on a business
# MAGIC key, and MERGEs (upserts) into a Silver table. Source, target, key
# MAGIC columns and dedupe rules are all parameterized.

# COMMAND ----------

#Testing git

dbutils.widgets.text("source_catalog", "main")
dbutils.widgets.text("source_schema", "bronze")
dbutils.widgets.text("source_table", "customers")
dbutils.widgets.text("target_catalog", "main")
dbutils.widgets.text("target_schema", "silver")
dbutils.widgets.text("target_table", "customers")
dbutils.widgets.text("primary_keys", "customer_id")        # comma-separated
dbutils.widgets.text("dedupe_order_column", "_ingested_at")  # keep latest row per key

source_catalog       = dbutils.widgets.get("source_catalog")
source_schema        = dbutils.widgets.get("source_schema")
source_table         = dbutils.widgets.get("source_table")
target_catalog       = dbutils.widgets.get("target_catalog")
target_schema        = dbutils.widgets.get("target_schema")
target_table         = dbutils.widgets.get("target_table")
primary_keys         = [c.strip() for c in dbutils.widgets.get("primary_keys").split(",")]
dedupe_order_column  = dbutils.widgets.get("dedupe_order_column")

source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
target_fqn = f"{target_catalog}.{target_schema}.{target_table}"

print(f"Source table : {source_fqn}")
print(f"Target table : {target_fqn}")
print(f"Keys         : {primary_keys}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Optional metadata lookup
# MAGIC As with Bronze, keys / column mappings / cleansing rules could instead
# MAGIC be pulled from `main.control.pipeline_metadata` (e.g. a JSON column
# MAGIC listing rename/cast rules per target table) so this notebook needs no
# MAGIC code changes when a new table is onboarded — only a new metadata row.

# COMMAND ----------

metadata_df = (
    spark.table("main.control.pipeline_metadata")
    .filter(f"layer = 'silver' AND target_table = '{target_table}'")
)

if metadata_df.count() > 0:
    row = metadata_df.collect()[0].asDict()
    primary_keys = row.get("primary_keys", ",".join(primary_keys)).split(",")
    dedupe_order_column = row.get("dedupe_order_column", dedupe_order_column)
    source_fqn = row.get("source_fqn", source_fqn)
    target_fqn = row.get("target_fqn", target_fqn)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read + cleanse

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(source_fqn)

# Standard cleansing: trim strings, drop rows missing any primary key
string_cols = [f.name for f in bronze_df.schema.fields if str(f.dataType) == "StringType()"]
for c in string_cols:
    bronze_df = bronze_df.withColumn(c, F.trim(F.col(c)))

bronze_df = bronze_df.dropna(subset=primary_keys)

# Dedupe: keep the latest record per business key
window_spec = Window.partitionBy(*primary_keys).orderBy(F.col(dedupe_order_column).desc())
silver_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter("_rn = 1")
    .drop("_rn")
    .withColumn("_processed_at", F.current_timestamp())
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Merge into Silver (upsert on primary key)

# COMMAND ----------

from delta.tables import DeltaTable

if spark.catalog.tableExists(target_fqn):
    target = DeltaTable.forName(spark, target_fqn)
    merge_condition = " AND ".join([f"tgt.{k} = src.{k}" for k in primary_keys])

    (
        target.alias("tgt")
        .merge(silver_df.alias("src"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged into existing Silver table: {target_fqn}")
else:
    silver_df.write.format("delta").saveAsTable(target_fqn)
    print(f"Created new Silver table: {target_fqn}")
